In [92]:
! pip install pandas numpy scikit-learn

In [93]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

In [94]:
# 1. Load Dataset
data = pd.read_csv('student_exam_data.csv')

In [95]:
# 2. Select Features and Target
X = data[["Study Hours", "Previous Exam Score"]].values
y = data["Pass/Fail"].values

In [96]:
# 3. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [97]:
# 4. Feature Scaling (MANDATORY for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [98]:
# MANUAL LOGISTIC REGRESSION
# Sigmoid Function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [99]:
# Initialize Parameters
weights = np.zeros(X_train_scaled.shape[1])
bias = 0
learning_rate = 0.01
epochs = 1000

In [100]:
# Gradient Descent (Manual Training)
for _ in range(epochs):
    linear_model = np.dot(X_train_scaled, weights) + bias
    y_pred = sigmoid(linear_model)

    dw = (1 / len(y_train)) * np.dot(X_train_scaled.T, (y_pred - y_train))
    db = (1 / len(y_train)) * np.sum(y_pred - y_train)

    weights -= learning_rate * dw
    bias -= learning_rate * db

In [101]:
# Manual Prediction & Thresholding
y_test_pred_prob_manual = sigmoid(np.dot(X_test_scaled, weights) + bias)
y_test_pred_manual = (y_test_pred_prob_manual >= 0.5).astype(int)

In [102]:
# Manual Confusion Matrix
TP = FP = TN = FN = 0

for actual, predicted in zip(y_test, y_test_pred_manual):
    if actual == 1 and predicted == 1:
        TP += 1
    elif actual == 0 and predicted == 0:
        TN += 1
    elif actual == 0 and predicted == 1:
        FP += 1
    elif actual == 1 and predicted == 0:
        FN += 1

confusion_matrix_manual = np.array([[TN, FP], [FN, TP]])

# Manual Accuracy Calculation
accuracy_manual = (TP + TN) / (TP + TN + FP + FN)

# Manual Precision, Recall, F1-score
precision_0 = TN / (TN + FN) if (TN + FN) != 0 else 0
recall_0 = TN / (TN + FP) if (TN + FP) != 0 else 0
f1_0 = (2 * precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) != 0 else 0

precision_1 = TP / (TP + FP) if (TP + FP) != 0 else 0
recall_1 = TP / (TP + FN) if (TP + FN) != 0 else 0
f1_1 = (2 * precision_1 * recall_1) / (precision_1 + recall_1) if (precision_1 + recall_1) != 0 else 0

# Manual Classification Report
support_0 = TN + FP
support_1 = TP + FN
total = support_0 + support_1

macro_precision = (precision_0 + precision_1) / 2
macro_recall = (recall_0 + recall_1) / 2
macro_f1 = (f1_0 + f1_1) / 2

weighted_precision = ((precision_0 * support_0) + (precision_1 * support_1)) / total
weighted_recall = ((recall_0 * support_0) + (recall_1 * support_1)) / total
weighted_f1 = ((f1_0 * support_0) + (f1_1 * support_1)) / total

In [103]:
# Manual Model Metrics
print("===== MANUAL LOGISTIC REGRESSION =====")
print("Accuracy (Manual):", round(accuracy_manual, 2))

print("\nConfusion Matrix (Manual):\n", confusion_matrix_manual)

print("\nClassification Report (Manual):\n")
print(f"{'Class':<10}{'Precision':<12}{'Recall':<10}{'F1-score':<10}{'Support':<10}")
print("-" * 55)
print(f"{'0':<10}{precision_0:<12.2f}{recall_0:<10.2f}{f1_0:<10.2f}{support_0:<10}")
print(f"{'1':<10}{precision_1:<12.2f}{recall_1:<10.2f}{f1_1:<10.2f}{support_1:<10}")
print("-" * 55)
print(f"{'Accuracy':<34}{accuracy_manual:<10.2f}{total:<10}")
print(f"{'Macro Avg':<10}{macro_precision:<12.2f}{macro_recall:<10.2f}{macro_f1:<10.2f}{total:<10}")
print(f"{'Weighted Avg':<10}{weighted_precision:<12.2f}{weighted_recall:<10.2f}{weighted_f1:<10.2f}{total:<10}")

===== MANUAL LOGISTIC REGRESSION =====
Accuracy (Manual): 0.83

Confusion Matrix (Manual):
 [[117  17]
 [ 17  49]]

Classification Report (Manual):

Class     Precision   Recall    F1-score  Support   
-------------------------------------------------------
0         0.87        0.87      0.87      134       
1         0.74        0.74      0.74      66        
-------------------------------------------------------
Accuracy                          0.83      200       
Macro Avg 0.81        0.81      0.81      200       
Weighted Avg0.83        0.83      0.83      200       


In [104]:
# SKLEARN LOGISTIC REGRESSION
# Train Logistic Regression Model
log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train)

LogisticRegression()

In [105]:
# 6. sklearn Prediction & Threshold
y_test_pred_prob_sklearn = log_model.predict_proba(X_test_scaled)[:, 1]
y_test_pred_sklearn = (y_test_pred_prob_sklearn >= 0.5).astype(int)

In [106]:
# 7. sklearn Model Metrics
print("===== SKLEARN LOGISTIC REGRESSION =====")
print("Accuracy:", accuracy_score(y_test, y_test_pred_sklearn))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_test_pred_sklearn))

print("\nClassification Report:\n", classification_report(y_test, y_test_pred_sklearn))

===== SKLEARN LOGISTIC REGRESSION =====
Accuracy: 0.83

Confusion Matrix:
 [[117  17]
 [ 17  49]]

Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.87      0.87       134
           1       0.74      0.74      0.74        66

    accuracy                           0.83       200
   macro avg       0.81      0.81      0.81       200
weighted avg       0.83      0.83      0.83       200



In [107]:
# Runtime Test Input
study_hours_input = float(input("Enter No. of study hours: "))
previous_marks_input = float(input("Enter previous exam marks: "))

runtime_input = np.array([[study_hours_input, previous_marks_input]])
runtime_input_scaled = scaler.transform(runtime_input)

In [108]:
# Runtime Prediction Output
# Manual Model
manual_prob = sigmoid(np.dot(runtime_input_scaled, weights) + bias)[0]
manual_result = "Pass" if manual_prob >= 0.5 else "Fail"

# sklearn Model
sklearn_prob = log_model.predict_proba(runtime_input_scaled)[0][1]
sklearn_result = "Pass" if sklearn_prob >= 0.5 else "Fail"

print("===== RUNTIME PREDICTION USING BOTH MODELS =====\n")
print(f"Prediction Results for {study_hours_input} Hour(s) of Study:")
print("-----------------------------------")
print(f"Manual Logistic Regression Prediction  : {manual_result} ({manual_prob:.2f})")
print(f"sklearn Logistic Regression Prediction : {sklearn_result} ({sklearn_prob:.2f})")
print("-----------------------------------")

===== RUNTIME PREDICTION USING BOTH MODELS =====

Prediction Results for 10.0 Hour(s) of Study:
-----------------------------------
Manual Logistic Regression Prediction  : Pass (0.94)
sklearn Logistic Regression Prediction : Pass (1.00)
-----------------------------------
